# Semana 7: Mini-hackathon de análisis exploratorio

**Introducción a la Ciencia de Datos** · Dudbil Olvasada Pabón Riaño · UNAB

Cuaderno guiado. Cada celda corresponde a un ejercicio de la guía de laboratorio; los resultados deben coincidir con sus puntos de control. Los datos se leen desde el repositorio del curso.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
BASE = "https://raw.githubusercontent.com/dudbilopr/Introduccion-a-la-ciencia-de-datos/main/Semanas/Datos/"

## 1. Leer el mapa
Cargue los datos y revise forma, tipos y vacíos. Compruebe qué fechas tiene cada estación: el Readme las nombra mal.

In [ ]:
df = pd.read_csv(BASE + "bicicletas_dia.csv", parse_dates=["dteday"])
print(df.shape, "vacíos:", df.isna().sum().sum())
df["temp_c"] = df["temp"] * 41
print(df.groupby("season")["dteday"].agg(["min", "max"]))
print("cnt = casual + registered:", (df["casual"] + df["registered"] == df["cnt"]).all())
print("Temperatura media:", round(df["temp_c"].mean(), 1), "°C; mínimo", round(df["temp_c"].min(), 1), "; máximo", round(df["temp_c"].max(), 1))

## 2. Separar por grupo
Antes de correr, escriba su apuesta: ¿se alquila más en días laborales?

In [ ]:
print(df.groupby("workingday")[["cnt", "casual", "registered"]].mean().round(0))
cuota = df.groupby("workingday")["casual"].sum() / df.groupby("workingday")["cnt"].sum()
print("Cuota de casuales:", cuota.round(3).to_dict())

## 3. Covarianza y correlación
Primero con la fórmula, luego con pandas.

In [ ]:
x, y = df["temp_c"], df["cnt"]
cov_mano = ((x - x.mean()) * (y - y.mean())).sum() / (len(x) - 1)
r_mano = cov_mano / (x.std() * y.std())
print("Covarianza a mano:", round(cov_mano, 1), " pandas:", round(x.cov(y), 1))
print("Correlación a mano:", round(r_mano, 3), " pandas:", round(x.corr(y), 3))
print("Humedad:", round((df["hum"] * 100).corr(y), 3), " Viento:", round((df["windspeed"] * 67).corr(y), 3))
plt.scatter(x, y, c=df["yr"], cmap="coolwarm", s=12)
plt.xlabel("Temperatura (°C)"); plt.ylabel("Alquileres por día"); plt.title("Más calor, más bicicletas, hasta los 30 °C")
plt.show()

## 4. La relación se curva

In [ ]:
rangos = pd.cut(df["temp_c"], [0, 5, 10, 15, 20, 25, 30, 35, 41])
print(df.groupby(rangos, observed=True)["cnt"].agg(["count", "mean"]).round(0))
anual = df.groupby("yr")["cnt"].sum()
print("2011:", anual[0], " 2012:", anual[1], " crecimiento:", round((anual[1] / anual[0] - 1) * 100, 1), "%")

## 5. Lo raro depende del contexto

In [ ]:
q1, q3 = df["cnt"].quantile(.25), df["cnt"].quantile(.75)
lim = q1 - 1.5 * (q3 - q1)
print("Q1", q1, "Q3", q3, "límite", lim, "atípicos:", (df["cnt"] < lim).sum())
print(df.loc[df["cnt"].idxmin(), ["dteday", "cnt"]].to_dict())
g = df.groupby(["yr", "mnth"])["cnt"]
q1m, q3m = g.transform(lambda s: s.quantile(.25)), g.transform(lambda s: s.quantile(.75))
raros = df[df["cnt"] < q1m - 1.5 * (q3m - q1m)]
print("Atípicos dentro de su mes:", len(raros))
print(raros.loc[raros["dteday"] >= "2012-10-01", ["dteday", "cnt"]])

## 6. Galería
Haga la gráfica de su pitch. El título dice la conclusión.

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(15, 4))
ax[0].hist(df["cnt"], bins=18); ax[0].set_title("¿Cómo se reparten los días?")
df.boxplot(column="cnt", by="season", ax=ax[1]); ax[1].set_title("1 = invierno según las fechas")
ax[2].plot(df["dteday"], df["cnt"]); ax[2].set_title("El sistema creció 64,9 % en un año")
plt.suptitle(""); plt.tight_layout(); plt.show()

## Declaración de uso de IA

Si usó una herramienta de IA, escriba aquí: cuál, para qué, el prompt, la versión del modelo y qué verificó usted.